# Package

In [28]:
import importlib

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} is not installed. Installing via !pip ...")
        try:
            get_ipython().system(f"pip install {package_name}")
            print(f"{package_name} installed successfully.")
        except Exception as e:
            print(f"Failed to install {package_name}. Error:\n{e}")

packages = ["pyDOE2", "diversipy", "pygmo", "optproblems", "pymoo"]

for pkg in packages:
    ensure_package(pkg)

from google.colab import drive
drive.mount('/content/drive')

pyDOE2 is already installed.
diversipy is already installed.
pygmo is not installed. Installing via !pip ...
  Using cached pygmo-v2.19.0.tar.gz (3.0 MB)
ERROR: pygmo from https://files.pythonhosted.org/packages/e2/12/090ba61479f60d5177a0048736d09dc028b2d65063ed44cb952df506336f/pygmo-v2.19.0.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
pygmo installed successfully.
optproblems is already installed.
pymoo is already installed.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
from pyDOE2 import lhs
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import time
import sys
import os
sys.path.append(os.path.abspath('/content/drive/MyDrive/PhD 2025-1 Offline data-driven MOP under uncertainty/ Prob-RVEA and Prob-MOEA D 2022'))

from desdeo_problem.Problem import DataProblem
from desdeo_problem.surrogatemodels.SurrogateKriging import SurrogateKriging
from desdeo_problem.testproblems.TestProblems import test_problem_builder
from desdeo_emo.EAs.ProbRVEA import ProbRVEA_v3
from pymoo.problems import get_problem
from pymoo.operators.sampling.lhs import LHS
from pymoo.problems.multi.omnitest import OmniTest

# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Problem

In [37]:
def mean_std(arr):
    return np.mean(arr), np.std(arr)

# Initial settings
# Problem: dtlz1-7, omnitest

problem_name = 'dtlz1'
problem_testbench = 'dtlz'

if 'dtlz' in problem_name:
  nvars = 10
  nobjs = 2
  problem_pymoo = get_problem(problem_name, n_var=nvars, n_obj=nobjs)

elif 'omnitest' in problem_name:
  nvars = 2
  nobjs = 2
  problem_pymoo = OmniTest(n_var=nvars)

#nsamples = 11*nvars-1
nsamples = 1000

print(f"Problem name: {problem_name}")
print(f"Cons: {problem_pymoo.n_constr}")
print(f"Var: {nvars}")
print(f"Obj: {nobjs}")


# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

Problem name: dtlz1
Cons: 0
Var: 10
Obj: 2

Min-Max normalization -> Min:  [0 0]
Min-Max normalization -> Max:  [552.3  568.36]
HV Reference points:  [1.1 1.1]


# Other

In [40]:
# Read dataset
np.random.seed(42)
sampling_pymoo = LHS()
x = sampling_pymoo(problem_pymoo, nsamples, seed=42).get("X")
y = problem_pymoo.evaluate(x, return_values_of=["F"])

print('problem_name', problem_name)
print('x', x.shape)
print('y', y.shape)

problem_name dtlz1
x (1000, 10)
y (1000, 2)


In [43]:
def to_matlab_matrix(X):
    rows = []
    for row in X:
        rows.append(" ".join(f"{v:.4f}" for v in row))
    return "[{}]".format(";\n ".join(rows))

print(to_matlab_matrix(x))


[0.0061 0.8358 0.5127 0.6455 0.5660 0.3325 0.6174 0.2338 0.4149 0.5603;
 0.1776 0.3345 0.8689 0.9118 0.1910 0.7638 0.4787 0.0017 0.2028 0.9833;
 0.4910 0.0215 0.9666 0.4511 0.7844 0.4420 0.3312 0.2206 0.2878 0.8317;
 0.4782 0.9162 0.3957 0.3337 0.5432 0.0024 0.0333 0.2511 0.9750 0.4478;
 0.0367 0.5501 0.8743 0.6893 0.6639 0.8402 0.4281 0.0680 0.5061 0.1895;
 0.8552 0.2802 0.5356 0.8911 0.2622 0.4276 0.5739 0.0509 0.2546 0.8658;
 0.0596 0.1504 0.7346 0.4342 0.3033 0.4104 0.6476 0.1530 0.3157 0.0580;
 0.9747 0.6744 0.8909 0.9859 0.4557 0.8570 0.8032 0.5266 0.8763 0.1041;
 0.9409 0.9523 0.5133 0.7067 0.1530 0.6774 0.1785 0.6460 0.4820 0.9511;
 0.9485 0.6427 0.1069 0.6879 0.9745 0.1895 0.3651 0.0663 0.3281 0.3533;
 0.0964 0.8773 0.6435 0.9661 0.0614 0.4570 0.3577 0.4216 0.4212 0.7194;
 0.9719 0.9433 0.5184 0.4309 0.4438 0.4171 0.8923 0.0249 0.0984 0.3432;
 0.4496 0.5200 0.2154 0.3425 0.2010 0.1296 0.4272 0.9874 0.7875 0.1323;
 0.1319 0.9509 0.0392 0.2194 0.3346 0.9293 0.5509 0.7469 0.3671 

In [33]:
# Create problem object
is_data = True
x_names = [f'x{i}' for i in range(1,nvars+1)]
y_names = [f'f{i}' for i in range(1,nobjs+1)]
row_names = ['lower_bound','upper_bound']
if is_data is False:
    prob = test_problem_builder(problem_name, nvars, nobjs)

    x = lhs(nvars, nsamples)
    y = prob.evaluate(x)
    data = pd.DataFrame(np.hstack((x,y.objectives)), columns=x_names+y_names)
else:
    data = pd.DataFrame(np.hstack((x,y)), columns=x_names+y_names)
if problem_testbench == 'DDMOPP':
    x_low = np.ones(nvars)*-1
    x_high = np.ones(nvars)
else:
    x_low = problem_pymoo.xl
    x_high = problem_pymoo.xu
    print('x_low: ', x_low)
    print('x_high: ', x_high)


bounds = pd.DataFrame(np.vstack((x_low,x_high)), columns=x_names, index=row_names)
problem = DataProblem(data=data, variable_names=x_names, objective_names=y_names,bounds=bounds)
start = time.time()
problem.train(SurrogateKriging)
end = time.time()
time_taken = end - start
print("Kriging surrogates built in:",time_taken,"(secs)")

x_low:  [0. 0.]
x_high:  [6. 6.]
Kriging surrogates built in: 6.851844549179077 (secs)


# Probabilstic RVEA

In [34]:
mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, nvars), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, :nobjs-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem_pymoo.evaluate(X_opt)
else:
    pf = problem_pymoo.pareto_front()
igd_plus = IGDPlus(pf)


for seed in range(1,31):
    # Run Probabilstic RVEA (fast)

    start_time = time.time()
    evolver_opt = ProbRVEA_v3(problem,
                              use_surrogates=True,
                              population_size = 100,
                              total_function_evaluations=10000)
    while evolver_opt.continue_evolution():
            evolver_opt.iterate()
    end_time = time.time()

    # Results
    solution = evolver_opt.population.individuals
    obj = evolver_opt.population.objectives
    f_real = problem_pymoo.evaluate(solution, return_values_of=["F"])
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    print(f"\nSeed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"IGD+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f}")


adapting
adapting
adapting

Seed 1 | Time: 90.88s | MSE: 2.13e-03 | IGD+: 1.14e-01 | Sur HV: 1.12 | Real HV: 1.12
adapting
adapting
adapting

Seed 2 | Time: 90.65s | MSE: 7.09e-04 | IGD+: 1.09e-01 | Sur HV: 1.14 | Real HV: 1.13
adapting
adapting
adapting

Seed 3 | Time: 90.36s | MSE: 1.63e-02 | IGD+: 7.58e-02 | Sur HV: 1.14 | Real HV: 1.14
adapting
adapting
adapting

Seed 4 | Time: 89.82s | MSE: 2.12e-02 | IGD+: 2.51e-01 | Sur HV: 1.11 | Real HV: 1.10
adapting
adapting
adapting

Seed 5 | Time: 90.57s | MSE: 3.16e-03 | IGD+: 9.40e-02 | Sur HV: 1.14 | Real HV: 1.14
adapting
adapting
adapting

Seed 6 | Time: 89.87s | MSE: 3.00e-02 | IGD+: 1.41e-01 | Sur HV: 1.14 | Real HV: 1.12
adapting
adapting
adapting

Seed 7 | Time: 91.48s | MSE: 6.22e-03 | IGD+: 1.05e-01 | Sur HV: 1.14 | Real HV: 1.14
adapting
adapting
adapting

Seed 8 | Time: 89.88s | MSE: 1.10e-02 | IGD+: 1.18e-01 | Sur HV: 1.13 | Real HV: 1.13
adapting
adapting
adapting

Seed 9 | Time: 90.53s | MSE: 2.59e-02 | IGD+: 7.46e-02 | Sur

Indicator results

In [35]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== Prob-RVEA: Statistics over 30 runs ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

Problem name:  omnitest

=== Prob-RVEA: Statistics over 30 runs ===
MSE: Mean = 1.16e-02, Std = 1.31e-02
IGD+: Mean = 1.45e-01, Std = 7.14e-02
Sur HV: Mean = 1.13, Std = 0.02
Real HV: Mean = 1.13, Std = 0.02


In [36]:
print(problem_name+'_MSE_Prob-RVEA'+' =',mse_list)
print(problem_name+'_IGD+_Prob-RVEA'+' =',igd_list)
print(problem_name+'_HV_Prob-RVEA'+' =',hv_real_list)

omnitest_MSE_Prob-RVEA = [0.002128523574757394, 0.0007093162899258379, 0.01628296364559311, 0.02122918458287583, 0.0031613370951101135, 0.03003831826441885, 0.006223603984331088, 0.01101409311659016, 0.025881862542200956, 0.005441924753363755, 0.03632092700253352, 0.0018028745098141402, 0.006653188435954028, 0.002395092116750375, 0.0050212078444280776, 0.0022272781371176203, 0.007807703131819489, 0.007236294292305731, 0.01167383942135162, 0.0022525283382383767, 0.024142013462557545, 0.01105999532145946, 0.007494616553928535, 0.0024629997997893355, 0.023255924324808882, 0.0017262413468233783, 0.003215124559471346, 0.0016999465427962278, 0.06018877860104988, 0.007687511073802679]
omnitest_IGD+_Prob-RVEA = [0.11424875590960909, 0.10887209876922119, 0.07575120087260138, 0.25103552812661417, 0.0939528578625281, 0.1410902607122091, 0.1045019352660763, 0.11791958385746523, 0.07458713339950802, 0.30671447757596537, 0.09748108795974678, 0.09874371983831214, 0.11530345668377283, 0.13862140664350